# SIAO-CNN-OBIGRU Training + Reliability Analysis (Colab Ready)

This notebook is designed for Google Colab execution of your modified NPPAD build.

- Architecture focus: **SIAO-CNN-OBIGRU** (Optimized Bi-GRU)
- Dataset focus: **13 active classes** (including simulated `TT`)
- Post-training analytics: dynamic reliability with λ, MTTF, and reliability curves
- Baseline comparison metrics: **RMSE, MAE, EVS, R²**

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def run_cmd(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout.strip())
    if proc.stderr:
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {cmd}")
    return proc


def ensure_git_lfs() -> None:
    if run_cmd("git lfs version", check=False).returncode != 0:
        run_cmd("apt-get -qq update")
        run_cmd("apt-get -qq install -y git-lfs")
    run_cmd("git lfs install")


def resolve_repo_root_colab() -> Path:
    candidates = [
        Path("/content/SIAO-CNN-ORNN"),
        Path("/content/drive/MyDrive/SIAO-CNN-ORNN"),
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / "train_pipeline.py").exists() and (candidate / "src").exists():
            return candidate

    target = Path("/content/SIAO-CNN-ORNN")
    clone_urls = [
        "https://github.com/sisoodiya/SIAO-CNN-ORNN.git",
        "https://github.com/sisoodiya/siao-cnn-ornn.git",
    ]

    if target.exists() and not (target / "train_pipeline.py").exists():
        raise FileExistsError(
            f"{target} already exists but is not a valid repo root. Remove it or set a different path."
        )

    for url in clone_urls:
        try:
            run_cmd(f"git clone {url} {target}")
            if (target / "train_pipeline.py").exists() and (target / "src").exists():
                return target
        except Exception as exc:
            print(f"Clone failed for {url}: {exc}")

    raise FileNotFoundError(
        "Unable to locate/clone repository. Upload the full repo to Colab or mount Drive and set path manually."
    )


if IN_COLAB:
    run_cmd("nvidia-smi", check=False)
    ensure_git_lfs()
    REPO_ROOT = resolve_repo_root_colab()
    os.chdir(REPO_ROOT)
    run_cmd("git lfs pull", check=False)
    run_cmd("python -m pip -q install imbalanced-learn optuna rich tqdm seaborn scipy scikit-learn")
else:
    cwd = Path.cwd().resolve()
    REPO_ROOT = cwd if (cwd / "train_pipeline.py").exists() else cwd.parent
    os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import explained_variance_score, mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
data_dir = REPO_ROOT / "data" / "Operation_csv_data"
if not data_dir.exists():
    raise FileNotFoundError(f"Dataset directory not found: {data_dir}")

class_counts = {}
for class_dir in sorted(data_dir.iterdir()):
    if class_dir.is_dir():
        class_counts[class_dir.name] = len(list(class_dir.glob("*.csv")))

active_counts = {k: v for k, v in class_counts.items() if v > 0}
total_samples = sum(active_counts.values())

print(f"Active classes: {len(active_counts)}")
print(f"Total samples: {total_samples}")
print(f"TT samples: {active_counts.get('TT', 0)}")

assert len(active_counts) == 13, f"Expected 13 active classes, got {len(active_counts)}"

dist_df = pd.DataFrame(
    [{"class": k, "samples": v} for k, v in active_counts.items()]
).sort_values("class").reset_index(drop=True)
dist_df

## Training Configuration

Set `FULL_TRAINING=False` for a quick smoke run, then switch back to `True` for full paper results.

In [ ]:
FULL_TRAINING = True

if FULL_TRAINING:
    CONFIG = {
        "data_dir": str(data_dir),
        "window_size": 100,
        "stride": 25,
        "cnn_embedding_dim": 256,
        "wks_dim": 97,
        "rnn_hidden_size": 224,
        "rnn_num_layers": 2,
        "num_classes": None,
        "wks_pop_size": 15,
        "wks_max_iter": 30,
        "siao_pop_size": 25,
        "siao_max_iter": 40,
        "bp_epochs": 100,
        "bp_lr": 0.00157,
        "fc_dropout": 0.164,
        "weight_decay": 1.97e-05,
        "batch_size": 163,
        "use_class_weights": True,
        "label_smoothing": 0.05,
        "use_smote": True,
        "balance_to_max": False,
        "smote_target_percentile": 50,
        "smote_min_samples": 6,
        "n_folds": 10,
        "random_seed": SEED,
        "use_cache": True,
        "cache_dir": str(REPO_ROOT / "data" / "processed_raw"),
        "save_dir": str(REPO_ROOT / "results"),
    }
else:
    CONFIG = {
        "data_dir": str(data_dir),
        "window_size": 100,
        "stride": 25,
        "cnn_embedding_dim": 128,
        "wks_dim": 97,
        "rnn_hidden_size": 96,
        "rnn_num_layers": 1,
        "num_classes": None,
        "wks_pop_size": 6,
        "wks_max_iter": 8,
        "siao_pop_size": 6,
        "siao_max_iter": 8,
        "bp_epochs": 5,
        "bp_lr": 0.00157,
        "fc_dropout": 0.164,
        "weight_decay": 1.97e-05,
        "batch_size": 64,
        "use_class_weights": True,
        "label_smoothing": 0.05,
        "use_smote": True,
        "balance_to_max": False,
        "smote_target_percentile": 50,
        "smote_min_samples": 6,
        "n_folds": 3,
        "random_seed": SEED,
        "use_cache": True,
        "cache_dir": str(REPO_ROOT / "data" / "processed_raw"),
        "save_dir": str(REPO_ROOT / "results"),
    }

for k, v in CONFIG.items():
    print(f"{k}: {v}")

In [ ]:
from train_pipeline import run_complete_pipeline

results = run_complete_pipeline(**CONFIG)
results

In [ ]:
results_dir = REPO_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

paper_summary = {
    "avg_accuracy": float(results["avg_accuracy"]),
    "std_accuracy": float(results["std_accuracy"]),
    "avg_macro_f1": float(results["avg_macro_f1"]),
    "fold_accuracies": [float(x) for x in results["fold_accuracies"]],
    "fold_macro_f1": [float(x) for x in results["fold_macro_f1"]],
    "model_design": "SIAO-CNN-OBIGRU",
    "active_classes": len(results.get("label_encoder_classes", [])),
    "tt_simulated": True
}

summary_path = results_dir / "training_summary_obigru_13class.json"
summary_path.write_text(json.dumps(paper_summary, indent=2))
print(f"Saved: {summary_path}")

fold_ids = np.arange(1, len(paper_summary["fold_accuracies"]) + 1)
acc = np.array(paper_summary["fold_accuracies"]) * 100.0
f1 = np.array(paper_summary["fold_macro_f1"]) * 100.0

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(fold_ids, acc, marker="o")
plt.title("Fold Accuracy (%)")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(fold_ids, f1, marker="o", color="tab:orange")
plt.title("Fold Macro-F1 (%)")
plt.xlabel("Fold")
plt.ylabel("Macro-F1")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Dynamic Reliability Analysis

This section follows the paper equations:

1. Failure rate: `λ(t) = cumulative_failures / cumulative_time`
2. Mean Time To Failure: `MTTF(t) = 1 / λ(t)`
3. Reliability: `R(t) = exp(-T/MTTF(t))`

If a `Normal` class exists, failures are defined as `class != Normal`.
If not, the notebook falls back to a prediction-error proxy stream for reliability modeling.

In [ ]:
y_true = np.asarray(results.get("oof_y_true", []), dtype=np.int64)
y_pred = np.asarray(results.get("oof_y_pred", []), dtype=np.int64)
label_classes = results.get("label_encoder_classes", [])

if y_true.size == 0 or y_pred.size == 0:
    raise RuntimeError("OOF predictions are missing. Please rerun training with current train_pipeline.py")

if "Normal" in label_classes:
    normal_id = label_classes.index("Normal")
    true_failure_events = (y_true != normal_id).astype(np.int64)
    pred_failure_events = (y_pred != normal_id).astype(np.int64)
    failure_mode = f"Class-based failure stream with Normal class id={normal_id}"
else:
    true_failure_events = (y_true != y_pred).astype(np.int64)
    pred_failure_events = true_failure_events.copy()
    failure_mode = "Prediction-error proxy stream (Normal class not present)"

print(f"Failure mode: {failure_mode}")
print(f"Failure events count (target): {true_failure_events.sum()} / {len(true_failure_events)}")

time_step_hours = 1.0


def dynamic_reliability(failure_events: np.ndarray, dt_hours: float = 1.0):
    failure_events = np.asarray(failure_events, dtype=np.float64)
    t = np.arange(1, len(failure_events) + 1, dtype=np.float64) * dt_hours
    cumulative_failures = np.cumsum(failure_events)
    lambda_t = cumulative_failures / t
    lambda_t = np.clip(lambda_t, 1e-12, None)
    mttf_t = 1.0 / lambda_t
    reliability_t = np.exp(-t / mttf_t)
    return t, lambda_t, mttf_t, reliability_t


t, lambda_true, mttf_true, reliability_true = dynamic_reliability(true_failure_events, time_step_hours)
_, lambda_ours, mttf_ours, reliability_ours = dynamic_reliability(pred_failure_events, time_step_hours)

# Baseline-1: constant reliability (mean of target)
reliability_const = np.full_like(reliability_true, reliability_true.mean())

# Baseline-2: linear trend
coef = np.polyfit(t, reliability_true, deg=1)
reliability_linear = np.clip(np.polyval(coef, t), 0.0, 1.0)

# Baseline-3: moving-average failure-rate converted to reliability
window = min(24, len(pred_failure_events))
kernel = np.ones(window) / window
ma_failure_prob = np.convolve(pred_failure_events, kernel, mode="same")
lambda_ma = np.clip(ma_failure_prob / time_step_hours, 1e-12, None)
mttf_ma = 1.0 / lambda_ma
reliability_ma = np.exp(-t / mttf_ma)


def score_curve(y_ref: np.ndarray, y_hat: np.ndarray):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_ref, y_hat))),
        "MAE": float(mean_absolute_error(y_ref, y_hat)),
        "EVS": float(explained_variance_score(y_ref, y_hat)),
        "R2": float(r2_score(y_ref, y_hat))
    }


scores = {
    "SIAO-CNN-OBIGRU Reliability": score_curve(reliability_true, reliability_ours),
    "Baseline Constant": score_curve(reliability_true, reliability_const),
    "Baseline Linear": score_curve(reliability_true, reliability_linear),
    "Baseline MovingAverage": score_curve(reliability_true, reliability_ma),
}

score_df = pd.DataFrame(scores).T.sort_values(["RMSE", "MAE"], ascending=True)
score_df

In [ ]:
plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.plot(t, lambda_true, label="Target λ(t)", linewidth=2)
plt.plot(t, lambda_ours, label="Predicted λ(t)", linewidth=2, alpha=0.8)
plt.title("Dynamic Failure Rate λ(t)")
plt.xlabel("Operating Time (hours)")
plt.ylabel("Failure Rate")
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(t, mttf_true, label="Target MTTF(t)", linewidth=2)
plt.plot(t, mttf_ours, label="Predicted MTTF(t)", linewidth=2, alpha=0.8)
plt.title("Dynamic Mean Time To Failure")
plt.xlabel("Operating Time (hours)")
plt.ylabel("MTTF (hours)")
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(t, reliability_true, label="Target Reliability", linewidth=2)
plt.plot(t, reliability_ours, label="SIAO-CNN-OBIGRU", linewidth=2)
plt.plot(t, reliability_const, label="Constant Baseline", linestyle="--")
plt.plot(t, reliability_linear, label="Linear Baseline", linestyle="--")
plt.plot(t, reliability_ma, label="MA Baseline", linestyle="--")
plt.title("Dynamic Reliability Curves")
plt.xlabel("Operating Time (hours)")
plt.ylabel("Reliability R(t)")
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(2, 2, 4)
rmse_values = score_df["RMSE"].values
labels = score_df.index.tolist()
plt.barh(labels, rmse_values)
plt.title("RMSE Comparison (lower is better)")
plt.xlabel("RMSE")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

score_df

In [ ]:
reliability_summary = {
    "failure_mode": failure_mode,
    "time_step_hours": time_step_hours,
    "scores": {k: {m: float(v) for m, v in d.items()} for k, d in scores.items()},
    "final_failure_rate": float(lambda_ours[-1]),
    "final_mttf": float(mttf_ours[-1]),
    "final_reliability": float(reliability_ours[-1])
}

reliability_path = results_dir / "reliability_summary_obigru_13class.json"
reliability_path.write_text(json.dumps(reliability_summary, indent=2))
print(f"Saved: {reliability_path}")
reliability_summary

In [ ]:
if IN_COLAB:
    run_cmd("cd /content/SIAO-CNN-ORNN && zip -r results_obigru_colab.zip results")
    from google.colab import files
    files.download("/content/SIAO-CNN-ORNN/results_obigru_colab.zip")
else:
    print(f"Results are available at: {results_dir}")